In [222]:
%matplotlib ipympl

import sys
sys.path.append("../..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from dash import Dash, dcc, html, Input, Output, callback
import dash_bootstrap_components as dbc

from pu.feature_extractors.extractors import ViTExtractor, AutoencoderExtractor
from pu.data.loaders import CSVLoader, SingleCSVLoader, SingleCSVWithTestLoader, FullCSVLoader

from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from PIL import Image

In [2]:
def get_all_features():
    extractors = ['clip-ViT-L-14']

    dataset_params = {
        'ava': ['/srv/PU-dataset/unlabeled.csv', 'id', '/srv/PU-dataset/dataset_unlabeled'],
        'aadb_train': ['/srv/aadb/train.csv', 'path', '/srv/aadb'],
        'aadb_val': ['/srv/aadb/validation.csv', 'path', '/srv/aadb'],
        'aadb_test': ['/srv/aadb/testnew.csv', 'path', '/srv/aadb'],
        'laion_aes': ['/srv/PU-dataset/positive.csv', 'path', '/srv/PU-dataset/dataset_positive']
    }

    all_features = {}

    for extractor in extractors:
        for dataset in dataset_params:
            featureset_name = f"{extractor}__{dataset}"
            loader = FullCSVLoader(*dataset_params[dataset])
            feature_extractor = ViTExtractor(extractor_name=extractor, experiment_name=featureset_name)

            path_col = dataset_params[dataset][1]
            data = loader.load_data()
            features = feature_extractor.extract_features(data[path_col])

            df = pd.concat([data.rename(columns={path_col: 'path'}), features.drop(columns=["id"])], axis=1)
            all_features[featureset_name] = df

    all_features['clip-ViT-L-14__aadb'] = pd.concat([all_features[f'clip-ViT-L-14__aadb_{split}'] for split in ['train', 'val', 'test']])
    all_features['clip-ViT-L-14__aadb']['label'] = all_features['clip-ViT-L-14__aadb']['label'] * 10

    return all_features

In [225]:
def project_features(features, first='laion_aes', second='ava', third='aadb'):
    if os.path.exists("embeddings/embeddings.npy"):
        with open("embeddings/embeddings.npy", "rb") as f:
            features_tsne = np.load(f)
        with open("embeddings/y.npy", "rb") as f:
            y_all = np.load(f)
        with open("embeddings/scores.npy", "rb") as f:
            scores_all = np.load(f)

    else:
        votes_column_dict = {'laion_aes': 'AESTHETIC_SCORE', 'ava': 'VotesMean', 'aadb': 'label'}
        
        features_first = features[f'clip-ViT-L-14__{first}'].filter(regex='^__feature.*$').to_numpy()
        features_second = features[f'clip-ViT-L-14__{second}'].filter(regex='^__feature.*$').to_numpy()
        features_third = features[f'clip-ViT-L-14__{third}'].filter(regex='^__feature.*$').to_numpy()
        features_all = np.concatenate([features_first, features_second, features_third], axis=0)
    
        y_first = np.zeros(len(features_first))
        y_second = np.ones(len(features_second))
        y_third = np.ones(len(features_third)) + 1
        y_all = np.concatenate([y_first, y_second, y_third])
    
        scores_first = features[f'clip-ViT-L-14__{first}'][votes_column_dict[first]].to_numpy()
        scores_second = features[f'clip-ViT-L-14__{second}'][votes_column_dict[second]].to_numpy()
        scores_third = features[f'clip-ViT-L-14__{third}'][votes_column_dict[third]].to_numpy()
        scores_all = np.concatenate([scores_first, scores_second, scores_third])
    
        features_pca = PCA(n_components=50).fit_transform(features_all)
        features_tsne = TSNE(n_components=2).fit_transform(features_pca)

        with open("embeddings/embeddings.npy", "wb") as f:
            np.save(f, features_tsne)
        with open("embeddings/y.npy", "wb") as f:
            np.save(f, y_all)
        with open("embeddings/scores.npy", "wb") as f:
            np.save(f, scores_all)

    return features_tsne, y_all, scores_all

In [228]:
names = ['laion_aes', 'ava', 'aadb']
features = get_all_features()
paths = [features[f'clip-ViT-L-14__{dataset}']['path'] for dataset in names]
projected, labels, scores = project_features(features, *names)

In [202]:
def plot_features_scores(projected_features, labels, scores, names):
    fig, axes = plt.subplots(1, 3, figsize=(45,10))
    features_first, scores_first  = projected_features[labels == 0], scores[labels==0]
    features_second, scores_second = projected_features[labels == 1], scores[labels==1]
    features_third, scores_third = projected_features[labels == 2], scores[labels==2]

    all_features = [features_first, features_second, features_third]
    all_scores = [scores_first, scores_second, scores_third]
    all_cmaps = ['seismic', 'seismic', 'seismic']

    for i, (features, scores, title, cmap) in enumerate(zip(all_features, all_scores, names, all_cmaps)):
        subplot = axes[i].scatter(features[:,0], features[:,1], c=scores, cmap=cmap, s=0.1, vmin=0, vmax=10)
        plt.colorbar(subplot, ax=axes[i])
        axes[i].set_title(title.replace('_', ' ').upper(), fontsize=20)
        axes[i].set_xlim([-150,150])
        axes[i].set_ylim([-150,150])

    plt.savefig('figures/features_projected_scores.png')

def plot_features(projected_features, labels, scores, names):
    fig, axes = plt.subplots(1, 3, figsize=(24,8))
    features_first, scores_first  = projected_features[labels == 0], scores[labels==0]
    features_second, scores_second = projected_features[labels == 1], scores[labels==1]
    features_third, scores_third = projected_features[labels == 2], scores[labels==2]

    all_features = [features_first, features_second, features_third]
    all_colors = ['green', 'red', 'blue']

    for i, (features, title, color) in enumerate(zip(all_features, names, all_colors)):
        axes[i].scatter(features[:,0], features[:,1], c=color, s=0.1, alpha=0.05)
        axes[i].set_title(title.replace('_', ' ').upper(), fontsize=20)
        axes[i].set_xlim([-150,150])
        axes[i].set_ylim([-150,150])

    plt.savefig('figures/features_projected.png')

def plot_features_scores_interactive(projected_features, labels, scores, names, paths):
    fig = go.FigureWidget(make_subplots(rows=2, cols=3, subplot_titles=[title.replace('_', ' ').upper() for title in names], shared_yaxes=True, shared_xaxes=True))
    
    features_first, scores_first  = projected_features[labels == 0], scores[labels==0]
    features_second, scores_second = projected_features[labels == 1], scores[labels==1]
    features_third, scores_third = projected_features[labels == 2], scores[labels==2]

    all_features = [features_first, features_second, features_third]
    all_scores = [scores_first, scores_second, scores_third]
    plots = []

    # Image show callback
    def show_image_callback(trace, points, selector):
        if points.point_inds:
            subplot = points.trace_index
            img = Image.open(f'{paths[subplot][points.point_inds[0]]}')

            figure_found = False
            for sub in fig.data:
                if(sub.name == f'image_{subplot}'):
                    figure_found = True
                    sub.z = img

            if not figure_found:
                go_img = go.Image(z=img, name=f'image_{subplot}')
                fig.add_trace(go_img, row=2, col=subplot+1)
                    
    for i, (features, scores, title) in enumerate(zip(all_features, all_scores, names)):
        fig.add_trace(
            go.Scattergl(
                x=features[:,0], y=features[:,1], mode='markers', name=f"plot_{i}",
                marker=dict(
                    size=1,
                    color=scores,
                    coloraxis='coloraxis'
                ),
                text=scores, hoverinfo='text', customdata=scores, hovertemplate='<extra></extra>Score: %{customdata:.2f}'
            ),
            row=1, col=i+1
        )
        
        fig.update_xaxes(range=[-150, 150], row=1, col=i+1)
        fig.update_yaxes(range=[-150, 150], row=1, col=i+1, showticklabels=False)
        fig.update_yaxes(showticklabels=False, row=2, col=i+1)
        
    fig.update_layout(
        {
            "xaxis": {"matches": "x", "showticklabels": False},
            "xaxis2": {"matches": "x", "showticklabels": False},
            "xaxis3": {"matches": "x", "showticklabels": False},
            "xaxis4": {"matches": None, "showticklabels": False},
            "xaxis5": {"matches": None, "showticklabels": False},
            "xaxis6": {"matches": None, "showticklabels": False},
        }
    )
    fig.update_layout(coloraxis=dict(colorscale=px.colors.diverging.balance, cmin=0, cmax=10), showlegend=False)
    fig.update_layout(width=1800, height=1000)
    fig.layout.hovermode = 'closest'
    for trace in fig.data:
        trace.on_click(show_image_callback)

    app = Dash()
    app.layout = html.Div([
        dcc.Graph(figure=fig)
    ])
    
    app.run_server(debug=True, use_reloader=False)  # Turn off reloader if inside Jupyter

In [220]:
def plot_features_scores_interactive_dash(projected_features, labels, scores, names, paths):
    features_first, scores_first  = projected_features[labels == 0], scores[labels==0]
    features_second, scores_second = projected_features[labels == 1], scores[labels==1]
    features_third, scores_third = projected_features[labels == 2], scores[labels==2]

    all_features = [features_first, features_second, features_third]
    all_scores = [scores_first, scores_second, scores_third]
    plots = []

    for i, (features, scores, title) in enumerate(zip(all_features, all_scores, names)):
        fig = go.Figure(
            go.Scattergl(
                x=features[:,0], y=features[:,1], mode='markers', name=f"plot_{i}",
                marker=dict(
                    size=1,
                    color=scores,
                    coloraxis='coloraxis'
                ),
                text=scores, hoverinfo='text', customdata=scores, hovertemplate='<extra></extra>Score: %{customdata:.2f}'
            )
        )
        
        fig.update_xaxes(range=[-150, 150])
        fig.update_yaxes(range=[-150, 150], showticklabels=False)
        fig.update_layout(coloraxis=dict(colorscale=px.colors.diverging.balance, cmin=0, cmax=10), showlegend=False)
        plots.append(fig)

    app = Dash()
    app.layout = html.Div(className='row', children=[
        dbc.Row([
            dbc.Col([
                dcc.Graph(
                    id="laion",
                    figure=plots[0]
                )
            ]),
            dbc.Col([
                dcc.Graph(
                    id="ava",
                    figure=plots[1]
                )
            ]),
            dbc.Col([
                dcc.Graph(
                    id="aadb",
                    figure=plots[2]
                )
            ]),
        ]),
        dbc.Row([
            
        ])
    ])

    app.run(debug=True)

In [221]:
plot_features_scores_interactive_dash(projected, labels, scores, names, paths)

In [ ]:
plot_features_scores_interactive(projected, labels, scores, names, paths)

In [ ]:
plot_features(projected, labels, scores, first, second, third)

In [ ]:
plot_features_scores(projected, labels, scores, first, second, third)